# Harmonization A — every storm on the squall line's profile

**What this notebook tests.** The supercell was rerun with the squall line's literal
thermodynamic sounding (its own winds unchanged), so that all three storms sit on the
*paper* profile family — mean-sea-level pressure anchoring, the unbalanced stratosphere,
and the few-percent moisture error included. Only the supercell changes; the isolated
convection and squall line are the manuscript runs.

**Why it is the right test.** It removes every *differential* setup error between the
storms without touching the physics of any of them, at the cost of leaving a *common-mode*
error that all three now share equally. Since every claim in the manuscript is comparative,
a common-mode error is harmless in a way a differential one is not.

---

**The question this notebook answers.** Not "did the numbers change" — they will. The
question is whether the setup differences bias the three storms *differently*, which is what
would make the manuscript's storm-to-storm comparisons unfair. So the load-bearing output is
not any individual figure but the cross-storm ratio table in §2.7: a setup error that moves
all three storms by the same factor leaves every comparative claim intact, while one that
moves them by different factors does not.

**Structure.**

| Part | Contents | Executed? |
|---|---|---|
| 1 | Initial-state characterization — how large the original setup errors are, and how much each harmonization removes | Yes |
| 2 | Results comparison — the manuscript's cross-storm figures and statistics, rendered once per arm | No — run this yourself |

Part 2 is left unexecuted deliberately: it needs a dask cluster, and it recomputes the
storm-development dataframes for the supercell only, which do not exist on disk yet.

## The two arms

Only `sc_wk` is rerun here. The isolated convection and squall line arms point at the same
directories in both arms, so those figures are identical by construction — they are still
rendered for both arms so each arm can be read as a complete, self-contained set.

## Imports and parameters

In [ ]:
# Native packages
from pathlib import Path
import sys
import warnings

# Other packages
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import HTML, display
from tqdm.notebook import tqdm

# Locate this notebook's own directory before importing the helper module.
# Jupyter Lab starts the kernel in the notebook's folder, but VS Code usually starts
# it in the workspace root instead, where `harmonization_utils` is not importable --
# so resolving it from the working directory alone fails depending on how you opened
# the notebook. Search the working directory, its parents, and their `validation/`
# subdirectories for the module rather than assuming any one of them.
_candidate_dirs = [
    Path.cwd(),
    Path.cwd() / "validation",
    *Path.cwd().parents,
    *[parent / "validation" for parent in Path.cwd().parents],
]
_validation_dir = next(
    (d for d in _candidate_dirs if (d / "harmonization_utils.py").is_file()), None
)
if _validation_dir is None:
    raise RuntimeError(
        "Could not find harmonization_utils.py from cwd "
        f"{Path.cwd()}. Set _validation_dir by hand to the repo's validation/ directory."
    )
if str(_validation_dir) not in sys.path:
    sys.path.insert(0, str(_validation_dir))
print(f"Using helper module from {_validation_dir}")

# Shared harmonization machinery. Importing this also puts the in-repo `cloudy`
# and `blt_utils_limited` packages on the path, so it must come before them.
import harmonization_utils as hu

# User packages
import cloudy as cm
import blt_utils_limited as blt

plt.style.use(blt.MPL_STYLE)
plt.rcParams["pdf.fonttype"] = 42.0

# Wider tables, since the comparison tables have many columns
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 200)

In [ ]:
# The two arms this notebook compares. "Original" is the set of runs the manuscript
# was written from; "harmonized" is this notebook's harmonization scheme.
ORIGINAL_ARM_LABEL = "original"
HARMONIZED_ARM_LABEL = "harmonized-sl-profile"

RUN_DIRS_BY_ARM = {
    ORIGINAL_ARM_LABEL: hu.PAPER_RUN_DIRS,
    HARMONIZED_ARM_LABEL: hu.HARMONIZED_ON_SL_PROFILE_RUN_DIRS,
}

# Where this notebook writes its figures and figure data. Deliberately not
# blt.FIGS_DIR, so a comparison run can never overwrite a manuscript figure.
OUTPUT_DIR = hu.HARMONIZATION_OUTPUT_ROOT / HARMONIZED_ARM_LABEL
FIGURE_OUTPUT_DIR = OUTPUT_DIR / "figures"
FIGURE_DATA_OUTPUT_DIR = OUTPUT_DIR / "figure_data"
for _output_dir in [FIGURE_OUTPUT_DIR, FIGURE_DATA_OUTPUT_DIR]:
    _output_dir.mkdir(parents=True, exist_ok=True)

# Show which run each storm resolves to in each arm, and flag the ones that differ,
# so the arms are unambiguous before any analysis runs.
_run_dir_rows = []
for _storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER:
    _original_run_dir = RUN_DIRS_BY_ARM[ORIGINAL_ARM_LABEL][_storm_key]
    _harmonized_run_dir = RUN_DIRS_BY_ARM[HARMONIZED_ARM_LABEL][_storm_key]
    _run_dir_rows.append(
        {
            "storm": blt.STORM_LABELS[_storm_key],
            ORIGINAL_ARM_LABEL: _original_run_dir.name,
            HARMONIZED_ARM_LABEL: _harmonized_run_dir.name,
            "rerun": _original_run_dir != _harmonized_run_dir,
        }
    )
display(pd.DataFrame(_run_dir_rows).set_index("storm"))

# Part 1 — Initial-state characterization

Everything here reads the **t=0 analysis file** of each run, never the RAMSIN. That is not
pedantry: one of the RAMSIN/output pairs in this study demonstrably disagrees (see the note
on the harmonized isolated-convection runs above), so the namelist is not a reliable record
of what a run started with. The initial state is horizontally homogeneous, so the
horizontal mean `xarrams` takes is exact rather than an approximation.

## 1.1 Read the t=0 soundings

In [ ]:
# Read the initial sounding for every storm in both arms. Runs shared between arms
# are read once and memoized, so this is cheap despite the apparent duplication.
#
# `calculate_sounding_derived_vars` runs one moist-adiabat parcel integration per
# starting level, which is why the CAPE profile is capped at parcels starting below
# 4 km; the integration itself always runs to the top of the column.
soundings_by_arm = {
    arm_label: hu.read_initial_soundings_for_arm(run_dirs)
    for arm_label, run_dirs in RUN_DIRS_BY_ARM.items()
}

# Relabel with human-readable storm names for plotting.
labelled_soundings_by_arm = {
    arm_label: {blt.STORM_LABELS[storm_key]: sounding_df
                for storm_key, sounding_df in soundings.items()}
    for arm_label, soundings in soundings_by_arm.items()
}

# Confirm every run sits on the same vertical grid, so the difference tables below
# compare like with like rather than silently interpolating across mismatched levels.
_all_sounding_dfs = [df for soundings in soundings_by_arm.values() for df in soundings.values()]
_reference_heights = _all_sounding_dfs[0]["z"].values
print(
    "All runs share the same vertical grid:",
    all(np.allclose(_reference_heights, df["z"].values) for df in _all_sounding_dfs),
)

## 1.2 The original setups: how large are the unintended differences?

The squall line and supercell were designed to have **identical** thermodynamics — same
temperature, same moisture, same pressure — differing only in their wind profiles. So every
non-zero entry in the squall-line row below is an unintended difference. The isolated
convection is deliberately drier, so its moisture and CAPE rows are expected to differ; its
pressure and temperature rows are not.

Two height windows are shown. The full column exposes the stratospheric potential-temperature
error; the `z ≤ 10 km` window is the depth the manuscript's analysis actually lives in, and is
the one that matters for whether the results are affected.

In [ ]:
# Difference every original storm against the supercell, over the full column and then
# over the analysis-relevant depth. The `z_of_max_abs_diff_m` column is the load-bearing
# one: a large potential-temperature difference is alarming at 5 km and irrelevant at 23 km,
# and only that column distinguishes the two cases.
for z_max, window_label in [(np.inf, "full column"), (10_000, "z <= 10 km")]:
    display(HTML(f"<h4>Original setups vs the supercell — {window_label}</h4>"))
    display(
        hu.sounding_difference_table(
            labelled_soundings_by_arm[ORIGINAL_ARM_LABEL],
            reference_label=blt.STORM_LABELS["sc_wk"],
            z_max=z_max,
        ).round(3)
    )

In [ ]:
# The moisture bug is a few-percent error, invisible on a profile overlay and obvious here.
# Sampled at 100 m: above any surface-layer artefact, well inside the well-mixed boundary
# layer where q_v is constant with height.
display(HTML("<h4>Low-level moisture: intended vs actual (original setups)</h4>"))
display(
    hu.low_level_moisture_table(
        soundings_by_arm[ORIGINAL_ARM_LABEL], hu.INTENDED_LOW_LEVEL_MIXING_RATIOS_G_KG
    ).round(4)
)

In [ ]:
# The pressure-anchoring error specifically. Anchoring one profile at 1000 hPa and another
# at mean sea level leaves potential temperature nearly identical while shifting actual
# temperature at a given height, so theta hides this error and T exposes it.
#
# Reading across the row shows whether the offset is a constant shift (a pure anchoring
# effect) or decays with height (the anchoring difference being absorbed by the lapse rate).
display(HTML("<h4>Temperature offset vs the supercell at fixed physical heights (original)</h4>"))
display(
    hu.temperature_offset_at_fixed_height_table(
        labelled_soundings_by_arm[ORIGINAL_ARM_LABEL],
        reference_label=blt.STORM_LABELS["sc_wk"],
    ).round(4)
)

In [ ]:
# Overlay all three original storms. Reading across a row shows *where* in the column the
# setups disagree; the narrowing height windows separate a stratosphere-only difference
# (visible only in the first column) from one that reaches the storm-bearing layer.
fig, axs = hu.plot_sounding_comparison(
    labelled_soundings_by_arm[ORIGINAL_ARM_LABEL],
    suptitle="Original setups — the three storms as the manuscript ran them",
)
cm.plotting.fig_multisave(fig, "p1_original_setups_soundings", FIGURE_OUTPUT_DIR)

## 1.3 The harmonized setups

The same tables for the harmonized on the squall line profile arm. What to look for: the squall-line and supercell
rows should collapse toward zero (they are supposed to be thermodynamically identical), and
the isolated convection's rows should retain only its *intended* moisture difference and the
CAPE that follows from it.

In [ ]:
for z_max, window_label in [(np.inf, "full column"), (10_000, "z <= 10 km")]:
    display(HTML(f"<h4>{HARMONIZED_ARM_LABEL} vs the supercell — {window_label}</h4>"))
    display(
        hu.sounding_difference_table(
            labelled_soundings_by_arm[HARMONIZED_ARM_LABEL],
            reference_label=blt.STORM_LABELS["sc_wk"],
            z_max=z_max,
        ).round(3)
    )

In [ ]:
display(HTML(f"<h4>Low-level moisture: intended vs actual ({HARMONIZED_ARM_LABEL})</h4>"))
display(
    hu.low_level_moisture_table(
        soundings_by_arm[HARMONIZED_ARM_LABEL], hu.INTENDED_LOW_LEVEL_MIXING_RATIOS_G_KG
    ).round(4)
)

display(HTML(f"<h4>Temperature offset vs the supercell at fixed heights ({HARMONIZED_ARM_LABEL})</h4>"))
display(
    hu.temperature_offset_at_fixed_height_table(
        labelled_soundings_by_arm[HARMONIZED_ARM_LABEL],
        reference_label=blt.STORM_LABELS["sc_wk"],
    ).round(4)
)

In [ ]:
fig, axs = hu.plot_sounding_comparison(
    labelled_soundings_by_arm[HARMONIZED_ARM_LABEL],
    suptitle=f"{HARMONIZED_ARM_LABEL} — the three storms after harmonization",
)
cm.plotting.fig_multisave(fig, "p1_harmonized_setups_soundings", FIGURE_OUTPUT_DIR)

## 1.4 The same comparisons on a pressure vertical coordinate

Everything above compares the storms at the same **physical height**, which is what makes the
pressure-anchoring error visible in the first place: anchoring one profile at 1000 hPa and
another at mean sea level shifts temperature by ~1 K at a given height even when the two
soundings are otherwise identical.

Replotting against **pressure** removes that offset by construction — two profiles built from
the same thermodynamic recipe but anchored differently will collapse onto each other here. So
any separation that *survives* this view is a real thermodynamic difference over and above the
anchoring, and any separation that *disappears* was the anchoring alone.

The squall line and supercell are the pair to watch: they were designed to be
thermodynamically identical, so on a pressure axis they should lie on top of one another.
Wherever they do not, something other than the anchoring is also different.

The height windows still filter on `z`, so each panel shows the same set of model levels as
its counterpart above and the two views can be read side by side.

In [ ]:
# Same overlays as sections 1.2 and 1.3, but with pressure as the vertical coordinate.
# The axis is inverted automatically so that low pressure (high altitude) is at the top.
for arm_label in RUN_DIRS_BY_ARM:
    fig, axs = hu.plot_sounding_comparison(
        labelled_soundings_by_arm[arm_label],
        y_variable="PS",
        suptitle=f"{arm_label} — profiles on a pressure vertical coordinate",
    )
    cm.plotting.fig_multisave(
        fig, f"p1_pressure_coordinate_soundings__{arm_label}", FIGURE_OUTPUT_DIR
    )

In [ ]:
# Quantify what the pressure view shows. Differencing on a *pressure* grid rather than a
# height grid is the numerical equivalent of the plot above: it removes the anchoring offset,
# so what remains is the thermodynamic difference on top of it.
#
# Compare against the height-based tables in 1.2 and 1.3 -- variables whose max|diff| collapses
# here were being driven by the anchoring alone.
def difference_table_on_pressure_levels(labelled_sounding_dfs, reference_label, variables):
    """Difference each sounding against a reference at matched *pressure* levels.

    Interpolates every sounding onto the reference's pressure levels before
    differencing, so the comparison is at constant pressure rather than constant
    height. np.interp needs its sample points increasing, and pressure decreases
    with height, so both arrays are reversed before interpolating.
    """
    reference_df = labelled_sounding_dfs[reference_label]
    reference_pressures = reference_df["PS"].values

    table_rows = []
    for label, sounding_df in labelled_sounding_dfs.items():
        if label == reference_label:
            continue
        for variable in variables:
            if variable not in sounding_df.columns:
                continue
            interpolated = np.interp(
                reference_pressures[::-1],
                sounding_df["PS"].values[::-1],
                np.asarray(sounding_df[variable].values, dtype=float)[::-1],
            )[::-1]
            difference = interpolated - np.asarray(
                reference_df[variable].values, dtype=float
            )
            finite = np.isfinite(difference)
            if not finite.any():
                continue
            absolute_difference = np.abs(np.where(finite, difference, np.nan))
            worst_ix = int(np.nanargmax(absolute_difference))
            table_rows.append(
                {
                    "comparison": label,
                    "variable": variable,
                    "max_abs_diff": float(np.nanmax(absolute_difference)),
                    "p_of_max_abs_diff_hPa": float(reference_pressures[worst_ix]),
                    "mean_abs_diff": float(np.nanmean(absolute_difference)),
                    "diff_at_surface": float(difference[np.argmax(finite)]),
                }
            )
    return pd.DataFrame(table_rows).set_index(["comparison", "variable"])


# PS is the coordinate now, so differencing it against itself carries no information.
_pressure_view_variables = [
    v for v in hu.SOUNDING_COMPARISON_VARIABLES if v != "PS"
]

for arm_label in RUN_DIRS_BY_ARM:
    display(HTML(f"<h4>{arm_label} vs the supercell, differenced at constant pressure</h4>"))
    display(
        difference_table_on_pressure_levels(
            labelled_soundings_by_arm[arm_label],
            reference_label=blt.STORM_LABELS["sc_wk"],
            variables=_pressure_view_variables,
        ).round(3)
    )

## 1.5 What changed for each storm

The tables above compare storms *within* an arm. This section instead compares each storm
against itself across the two arms, which is what tells you how much of a given storm's
result change (in Part 2) can be attributed to its initial state moving at all.

A storm whose two curves lie on top of each other was not rerun, and its Part 2 figures will
be identical between arms by construction.

In [ ]:
# One figure per storm: its original profile against its harmonized profile.
for storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER:
    storm_label = blt.STORM_LABELS[storm_key]
    original_sounding_df = soundings_by_arm[ORIGINAL_ARM_LABEL][storm_key]
    harmonized_sounding_df = soundings_by_arm[HARMONIZED_ARM_LABEL][storm_key]

    # Skip the overlay when the arm did not rerun this storm; an identical pair of
    # curves communicates nothing and just adds figures to scroll past.
    was_rerun = (
        RUN_DIRS_BY_ARM[ORIGINAL_ARM_LABEL][storm_key]
        != RUN_DIRS_BY_ARM[HARMONIZED_ARM_LABEL][storm_key]
    )
    if not was_rerun:
        print(f"{storm_label}: not rerun in this harmonization — initial state is identical")
        continue

    fig, axs = hu.plot_sounding_comparison(
        {ORIGINAL_ARM_LABEL: original_sounding_df,
         HARMONIZED_ARM_LABEL: harmonized_sounding_df},
        suptitle=f"{storm_label}: original vs harmonized initial state",
    )
    cm.plotting.fig_multisave(
        fig, f"p1_per_storm_change_{storm_key}", FIGURE_OUTPUT_DIR
    )

In [ ]:
# Quantify the same thing: for each storm, how far did its own initial state move?
for storm_key in hu.STORM_KEYS_IN_FIGURE_ORDER:
    storm_label = blt.STORM_LABELS[storm_key]
    display(HTML(f"<h4>{storm_label}: harmonized minus original (z &le; 10 km)</h4>"))
    display(
        hu.sounding_difference_table(
            {
                ORIGINAL_ARM_LABEL: soundings_by_arm[ORIGINAL_ARM_LABEL][storm_key],
                HARMONIZED_ARM_LABEL: soundings_by_arm[HARMONIZED_ARM_LABEL][storm_key],
            },
            reference_label=ORIGINAL_ARM_LABEL,
            z_max=10_000,
        ).round(3)
    )

## 1.6 Summary — does harmonization actually equalize the storms?

The single number that matters for fairness is the **spread across storms**, before and
after. Surface-based CAPE is the natural summary: it integrates the temperature, moisture
and pressure errors into one quantity, and it is the quantity most directly tied to how
vigorously each storm can convect.

Because the isolated convection is *meant* to be different, the fair comparison is the
squall line against the supercell — two storms designed to be thermodynamically identical.
Any gap between those two is pure setup error.

In [ ]:
# Surface-based CAPE, CIN and LCL height for every storm in both arms.
stability_rows = []
for arm_label, soundings in soundings_by_arm.items():
    for storm_key, sounding_df in soundings.items():
        stability_rows.append(
            {
                "arm": arm_label,
                "storm": blt.STORM_LABELS[storm_key],
                # Row 0 is the lowest model level, i.e. the surface-based parcel.
                "SBCAPE_J_kg": float(sounding_df["cape"].iloc[0]),
                "SBCIN_J_kg": float(sounding_df["cin"].iloc[0]),
                "LCL_z_m": float(sounding_df["lcl"].iloc[0]),
                "q_v_100m_g_kg": float(
                    np.interp(100.0, sounding_df["z"].values, sounding_df["q_v"].values)
                ),
            }
        )
stability_table = pd.DataFrame(stability_rows).set_index(["arm", "storm"])
display(HTML("<h4>Surface-based stability by arm</h4>"))
display(stability_table.round(2))

# The fairness number: the squall line and supercell are supposed to be identical, so
# their gap is pure setup error. Track how much each harmonization removes.
display(HTML("<h4>Squall line vs supercell — two storms designed to be identical</h4>"))
squall_supercell_rows = []
for arm_label in RUN_DIRS_BY_ARM:
    squall_line_cape = stability_table.loc[(arm_label, blt.STORM_LABELS["sl_wk"]), "SBCAPE_J_kg"]
    supercell_cape = stability_table.loc[(arm_label, blt.STORM_LABELS["sc_wk"]), "SBCAPE_J_kg"]
    squall_supercell_rows.append(
        {
            "arm": arm_label,
            "SL_SBCAPE": squall_line_cape,
            "SC_SBCAPE": supercell_cape,
            "difference_J_kg": squall_line_cape - supercell_cape,
            "percent_of_SC": 100.0 * (squall_line_cape - supercell_cape) / supercell_cape,
        }
    )
display(pd.DataFrame(squall_supercell_rows).set_index("arm").round(2))

# Part 2 — Results comparison

**This part has not been executed.** It needs a dask cluster, and it recomputes the
storm-development dataframe(s) for the supercell only, which are not on disk yet.
Run it yourself and send me any traceback.

### Where the derived products live

The paper runs and the harmonized runs keep their derived products in **different places**:
the paper runs' are in the in-repo tree `replication/data/derived/<storm>/` (what the
committed `blt_utils.py` points the manuscript notebook at), while each harmonized run keeps
its own in `<run_dir>/derived/`. `harmonization_utils` resolves this per run, so nothing here
needs `blt_utils.py` to be in any particular state — but it is worth knowing, because getting
it wrong would not raise, it would just silently recompute everything from raw output.

The precomputed tracer profile datasets exist for **every** run in both arms, so Figures 7
and 9 and the statistics tables need no recomputation. The development dataframes exist only
for the paper runs, so they are computed and cached on first use for the reruns.

Each figure below is rendered **once per arm**, stacked, exactly as
`validation_comparison.ipynb` does — the figure code itself is carried over essentially
verbatim, restructured into functions that take their data as arguments rather than reading
it from a stashed global namespace.

### One important caveat before you run this

Figure 12 and two of the three rows of Figure 11 depend on the **upward mass flux through
10 km**, which the manuscript computes from the 2-second lite files. Every harmonized run
was submitted with `FRQLITE = 0` and therefore wrote **no lite output at all**, so the
manuscript's exact flux is uncomputable for the harmonized arm.

The workaround here is to estimate the flux from the 5-minute analysis output for **both**
arms, using an identical construction, so the arm-to-arm comparison stays valid even though
the absolute values will not match the manuscript's. §2.4.1 quantifies that bias directly by
computing both estimators on the paper runs, which do have lite output. **Read that
validation cell before trusting the Figure 11/12 numbers**, and if the two estimators
disagree badly, treat those two figures as qualitative only.

## 2.0 Cluster and data loading

In [ ]:
# The 5-minute datasets are large enough that the reductions below want a cluster.
USE_DASK_LOCAL_CLUSTER = True

if USE_DASK_LOCAL_CLUSTER:
    from dask.distributed import Client, LocalCluster

    cluster = LocalCluster()
    client = Client(cluster)
    # client = Client(blt.DASK_SCHEDULER_ADDRESS)
    display(client)

In [ ]:
# Load, per arm, the two cheap things: the 5-minute analysis output (lazy, dask-backed)
# and the precomputed in-updraft tracer profiles.
#
# `blt_paths_for_arm` points blt's module-level path constants at this arm's runs for the
# duration of the block and restores them afterwards. That restoration matters -- blt_utils
# is imported by the manuscript notebook in this same repo, and leaving it mutated is how an
# analysis silently reads the wrong run.
storm_dss_by_arm = {}
tracer_totals_profile_dss_by_arm = {}
tracer_totals_dss_by_arm = {}

for arm_label, run_dirs in RUN_DIRS_BY_ARM.items():
    print(f"\n===== {arm_label} =====")
    with hu.blt_paths_for_arm(run_dirs, figure_data_dir=FIGURE_DATA_OUTPUT_DIR):
        # 5-minute RAMS output, postprocessed (derived thermodynamics, in_updraft,
        # in_anvil, in_analysis, and the number-concentration tracer variables).
        storm_dss_by_arm[arm_label] = hu.read_storm_datasets(run_dirs)

        # Precomputed in-updraft tracer vertical profiles. These already exist for every
        # run in both arms, so Figures 7 and 9 need no recomputation from raw output.
        (
            tracer_totals_profile_dss_by_arm[arm_label],
            tracer_totals_dss_by_arm[arm_label],
        ) = hu.read_tracer_totals_datasets(run_dirs)

# Output interval and grid spacing, which the flux estimator needs. Uniform across runs.
# Defined here rather than after the expensive step below, so that every later cell still
# has them even if the development-dataframe computation fails.
_any_storm_ds = next(iter(storm_dss_by_arm[ORIGINAL_ARM_LABEL].values()))
DELTA_T = (_any_storm_ds["time"].values[1] - _any_storm_ds["time"].values[0]) / np.timedelta64(1, "s")
DELTA_X = _any_storm_ds["x"].values[1] - _any_storm_ds["x"].values[0]
print(f"\nDELTA_T = {DELTA_T} s, DELTA_X = {DELTA_X} m")

### Development dataframes

Separated from the load above because this is the one genuinely expensive step in Part 2:
it reduces over the full `(time, z, y, x)` updraft, anvil and rainfall fields. The paper runs
have theirs cached already; the rerun storms are computed here and written into each run's
own `derived/` directory, so a second pass through this notebook is fast.

In [ ]:
# Development dataframes. Present on disk for the paper runs; computed and cached for the
# harmonized reruns.
development_dfs_by_arm = {}

for arm_label, run_dirs in RUN_DIRS_BY_ARM.items():
    print(f"\n===== {arm_label} =====")
    with hu.blt_paths_for_arm(run_dirs, figure_data_dir=FIGURE_DATA_OUTPUT_DIR):
        development_dfs_by_arm[arm_label] = hu.read_or_compute_development_dataframes(
            run_dirs, storm_dss_by_arm[arm_label]
        )

display(development_dfs_by_arm[HARMONIZED_ARM_LABEL]["sc_wk"].head())

## 2.1 Figure 7 — tracer entrainment

Entrained fraction of cumulatively emitted tracer (top block) and mean tracer mixing ratio
(bottom block), for fixed-source and rain-sourced tracers in each storm.

In [ ]:
def plot_figure_7_tracer_entrainment(arm_label, tracer_totals_dss):
    """Render Figure 7 for one arm.

    Carried over from `validation_comparison.ipynb` essentially verbatim; the only
    structural change is that the tracer datasets arrive as an argument instead of
    being restored from a stashed global namespace, and the figure name carries the
    arm label so the two arms write distinct files.
    """
    figure_name = f"f07_tracer_entrainment__{arm_label}"
    figure_data = {}

    fig = plt.figure(figsize=(12, 11))
    outer_subfigs = fig.subfigures(nrows=2, hspace=0.05)

    ########## Fractions ##########
    axs = outer_subfigs[0].subplots(nrows=2, ncols=3, sharex=True)
    outer_subfigs[0].subplots_adjust(left=0.13, hspace=0.5, top=0.87, right=0.82)

    for tracer_type_ix, tracer_type in enumerate(["Fixed-source", "Rain-sourced"]):
        for storm_ix, (storm, this_tracer_ds) in enumerate(tracer_totals_dss.items()):
            ax = axs[tracer_type_ix, storm_ix]
            if tracer_type_ix == 0:
                for ix, tracer in enumerate(blt.FS_TRACERS):
                    blt.plot_and_save(
                        figure_data,
                        f"storm-{storm}_tracer-{tracer}_frac_entrained_of_emitted",
                        ax.plot,
                        this_tracer_ds["t_minutes"],
                        this_tracer_ds[tracer + "_nc"] / this_tracer_ds["ACC" + tracer + "_nc"],
                        color=mpl.colormaps[blt.PROGRESSIVE_CMAP](ix / len(blt.FS_TRACERS)),
                    )
            else:
                for ix, tracer in enumerate(blt.RS_TRACERS):
                    blt.plot_and_save(
                        figure_data,
                        f"storm-{storm}_tracer-{tracer}_frac_entrained_of_emitted",
                        ax.plot,
                        this_tracer_ds["t_minutes"],
                        (this_tracer_ds[tracer + "_nc"] / this_tracer_ds["ACC" + tracer + "_nc"]).values,
                        color=mpl.colormaps[blt.PROGRESSIVE_CMAP](ix / len(blt.RS_TRACERS)),
                        label=str(blt.RS_TRACER_THRESHOLDS_MM_HR[ix]) + " mm/hr",
                    )

            ax.set_title(blt.axes_title(storm_ix + tracer_type_ix * 3, blt.STORM_LABELS[storm]))
            if storm_ix == 0:
                ax.set_ylabel("Fraction")
            blt.label_x_axis(ax)

        # Share y on the mean row; x is already shared across the subfigure.
        cm.plotting.share_axes(axs[1], x=False, y=True)

    outer_subfigs[0].suptitle("Entrained fraction of cumulatively emitted tracer", fontsize=22)

    ########## Means ##########
    axs = outer_subfigs[1].subplots(nrows=2, ncols=3, sharex=True, sharey="row")
    outer_subfigs[1].subplots_adjust(left=0.15, hspace=0.5, top=0.87, right=0.82)

    for tracer_type_ix, tracer_type in enumerate(["Fixed-source", "Rain-sourced"]):
        for storm_ix, (storm, this_tracer_ds) in enumerate(tracer_totals_dss.items()):
            ax = axs[tracer_type_ix, storm_ix]
            if tracer_type_ix == 0:
                for ix, tracer in enumerate(sorted(blt.FS_TRACERS)):
                    blt.plot_and_save(
                        figure_data,
                        f"storm-{storm}_tracer-{tracer}_nc",
                        ax.plot,
                        this_tracer_ds["t_minutes"],
                        this_tracer_ds[tracer + "_nc"] / this_tracer_ds["air_mass"],
                        color=mpl.colormaps[blt.PROGRESSIVE_CMAP](ix / len(blt.FS_TRACERS)),
                    )
            else:
                for ix, tracer in enumerate(blt.RS_TRACERS):
                    blt.plot_and_save(
                        figure_data,
                        f"storm-{storm}_tracer-{tracer}_nc",
                        ax.plot,
                        this_tracer_ds["t_minutes"],
                        (this_tracer_ds[tracer + "_nc"] / this_tracer_ds["air_mass"]).values,
                        color=mpl.colormaps[blt.PROGRESSIVE_CMAP](ix / len(blt.RS_TRACERS)),
                        label=str(blt.RS_TRACER_THRESHOLDS_MM_HR[ix]) + " mm/hr",
                    )

            # Rain-sourced mixing ratios span orders of magnitude
            if tracer_type_ix == 1:
                ax.set_yscale("log")
                ax.set_ylim([5e2, 2e5])
            ax.set_title(blt.axes_title(6 + storm_ix + tracer_type_ix * 3, blt.STORM_LABELS[storm]))
            if storm_ix == 0:
                ax.set_ylabel("#/kg")
            blt.label_x_axis(ax)

    outer_subfigs[1].suptitle("Mean tracer mixing ratio", fontsize=22)

    # Colorbars: emission time for the fixed-source tracers, a legend for rain-sourced
    for subfig in outer_subfigs:
        fs_cbar_ax = subfig.add_axes([0.83, 0.55, 0.02, 0.32])  # Left, bottom, width, height
        fs_cbar = mpl.colorbar.ColorbarBase(
            ax=fs_cbar_ax,
            cmap=blt.PROGRESSIVE_CMAP + "_r",
            norm=mpl.colors.Normalize(vmin=0, vmax=1),
            orientation="vertical",
        )
        # Ticks are a linear range over the 0-1 normalized emission time
        tick_labels = np.arange(0, 181, 30)
        tick_values = (tick_labels / 180)[::-1]
        fs_cbar.set_ticks(tick_values)
        fs_cbar.set_ticklabels(tick_labels)
        fs_cbar_ax.xaxis.set_label_position("top")
        fs_cbar_ax.set_xlabel("Emission time\n(minutes)")
        fs_cbar_ax.get_xaxis().labelpad = 10

        cm.plotting.clean_legend(
            subfig.axes[3],
            bbox_transform=subfig.transSubfigure,
            title="Rain threshold",
            bbox_to_anchor=[0.800, 0.468],
            fontsize=14,
            title_fontsize=16,
            loc="upper left",
        )

    # Row headers
    row_headers = {}
    for subfig_ix, subfig in enumerate(outer_subfigs):
        for row_ix in range(2):
            row_headers[(subfig_ix, row_ix)] = cm.plotting.add_row_header(
                subfig.axes[row_ix * 3],
                "Fixed-source" if row_ix == 0 else "Rain-sourced",
            )

    # The log scale on the second subfigure's lower row shifts its header; pull it back
    # into line with the header above it.
    reference_header = row_headers[(1, 0)]
    problematic_header = row_headers[(1, 1)]
    problematic_header.set_position(
        (reference_header.get_position()[0], problematic_header.get_position()[1])
    )

    fig.suptitle(f"[{arm_label}]", fontsize=13, y=0.995)

    blt.save_figure_data(figure_data, figure_name)
    cm.plotting.fig_multisave(fig, figure_name, FIGURE_OUTPUT_DIR, no_title_version=False)
    return fig


for arm_label in RUN_DIRS_BY_ARM:
    with hu.blt_paths_for_arm(RUN_DIRS_BY_ARM[arm_label], figure_data_dir=FIGURE_DATA_OUTPUT_DIR):
        plot_figure_7_tracer_entrainment(arm_label, tracer_totals_dss_by_arm[arm_label])

## 2.2 Figure 9 — rain-sourced tracer entrainment factors

The decomposition of rain-sourced entrainment into its factors: fixed-source mixing ratio
and updraft mass, rain-sourced fraction, rain production efficiency, and rain-sourced
entrainment efficiency. This is the figure that carries most of the manuscript's mechanistic
cross-storm argument, so it is the one most worth scrutinizing between arms.

In [ ]:
def plot_figure_9_rs_entrainment_factors(arm_label, tracer_totals_dss):
    """Render Figure 9 for one arm. Body carried over verbatim from the template."""
    figure_name = f"f09_rs_entrainment_factors__{arm_label}"
    figure_data = {}

    fig, axs = plt.subplots(nrows=4, ncols=3, figsize=(12, 11), sharex=True, sharey="row")
    fig.subplots_adjust(wspace=0.55, left=0.19, hspace=0.5, bottom=0)

    headers = []
    bbox_to_anchor = (0.5, -0.23)

    # Each row is its own function so it can be broken out into a standalone figure
    def rs_factors_fs_entrainment(axs, ax_title_start_ix=0):
        updraft_axs = []
        for storm_ix, (storm, this_tracer_totals_ds) in enumerate(tracer_totals_dss.items()):
            ax = axs[storm_ix]
            blt.plot_and_save(
                figure_data,
                f"storm-{storm}_fs_entrainment",
                ax.plot,
                this_tracer_totals_ds["t_minutes"],
                this_tracer_totals_ds["total_fs_tracer_nc"] / this_tracer_totals_ds["air_mass"],
                label=r"$\overline{q_{RS}}$",
                color=cm.plotting.get_nth_color(0),
            )
            # Empty line so the legend picks up the twin axis's series too
            ax.plot([], [], color=cm.plotting.get_nth_color(1), label="Updraft mass")
            updraft_ax = ax.twinx()
            updraft_axs.append(updraft_ax)
            blt.plot_and_save(
                figure_data,
                f"storm-{storm}_updraft_mass",
                updraft_ax.plot,
                this_tracer_totals_ds["t_minutes"],
                this_tracer_totals_ds["air_mass"],
                color=cm.plotting.get_nth_color(1),
            )
            updraft_ax.set_yscale("log")
            ax.set_title(blt.axes_title(ax_title_start_ix + storm_ix, blt.STORM_LABELS[storm]))
            ax.set_ylabel(r"$\#\ kg^{-1}$")
            blt.label_x_axis(ax)
            updraft_ax.set_ylabel("kg")

        cm.plotting.share_axes(updraft_axs, x=False, y=True)
        for ax in updraft_axs:
            ax.set_ylim([1e9, 2e14])
        axs[1].legend(loc="center", bbox_to_anchor=bbox_to_anchor, ncol=2)

    def rs_factors_rs_fraction(axs, ax_title_start_ix=0):
        for storm_ix, (storm, this_tracer_totals_ds) in enumerate(tracer_totals_dss.items()):
            ax = axs[storm_ix]
            for tracer_ix, tracer in enumerate(blt.RS_TRACERS):
                blt.plot_and_save(
                    figure_data,
                    f"storm-{storm}_rs_fraction",
                    ax.plot,
                    this_tracer_totals_ds["t_minutes"],
                    this_tracer_totals_ds[tracer + "_nc"] / this_tracer_totals_ds["total_fs_tracer_nc"],
                    label=str(blt.RS_TRACER_THRESHOLDS_MM_HR[tracer_ix]) + " mm/hr",
                    color=mpl.colormaps[blt.PROGRESSIVE_CMAP](tracer_ix / len(blt.RS_TRACERS)),
                )
            ax.set_title(blt.axes_title(ax_title_start_ix + storm_ix, blt.STORM_LABELS[storm]))
            if storm_ix == 0:
                ax.set_ylabel("Fraction")
            blt.label_x_axis(ax)
        axs[1].legend(loc="center", bbox_to_anchor=bbox_to_anchor, ncol=5)

    def rs_factors_rain_efficiency(axs, ax_title_start_ix=0, storms=None):
        storms = storms if storms is not None else tracer_totals_dss.keys()
        for storm_ix, storm in enumerate(storms):
            this_tracer_totals_ds = tracer_totals_dss[storm]
            ax = axs[storm_ix]
            for tracer_ix, tracer in enumerate(blt.RS_TRACERS):
                blt.plot_and_save(
                    figure_data,
                    f"storm-{storm}_rain_efficiency",
                    ax.plot,
                    this_tracer_totals_ds["t_minutes"],
                    this_tracer_totals_ds["ACC" + tracer + "_nc"] / this_tracer_totals_ds["air_mass"],
                    label=str(blt.RS_TRACER_THRESHOLDS_MM_HR[tracer_ix]) + " mm/hr",
                    color=mpl.colormaps[blt.PROGRESSIVE_CMAP](tracer_ix / len(blt.RS_TRACERS)),
                )
                ax.set_title(blt.axes_title(ax_title_start_ix + storm_ix, blt.STORM_LABELS[storm]))
                if storm_ix == 0:
                    ax.set_ylabel(r"$\#\ kg^{-1}$")
            ax.set_yscale("log")
            blt.label_x_axis(ax)
        axs[1].legend(loc="center", bbox_to_anchor=bbox_to_anchor, ncol=5)

    def rs_factors_rs_entrainment_efficiency(axs, ax_title_start_ix=0):
        for storm_ix, (storm, this_tracer_totals_ds) in enumerate(tracer_totals_dss.items()):
            ax = axs[storm_ix]
            for tracer_ix, tracer in enumerate(blt.RS_TRACERS):
                blt.plot_and_save(
                    figure_data,
                    f"storm-{storm}_rs_entrainment_efficiency",
                    ax.plot,
                    this_tracer_totals_ds["t_minutes"],
                    this_tracer_totals_ds[tracer + "_nc"] / this_tracer_totals_ds["ACC" + tracer + "_nc"],
                    label=str(blt.RS_TRACER_THRESHOLDS_MM_HR[tracer_ix]) + " mm/hr",
                    color=mpl.colormaps[blt.PROGRESSIVE_CMAP](tracer_ix / len(blt.RS_TRACERS)),
                )
                ax.set_title(blt.axes_title(9 + storm_ix, blt.STORM_LABELS[storm]))
                if storm_ix == 0:
                    ax.set_ylabel("Fraction")
                blt.label_x_axis(ax)
        axs[1].legend(
            loc="center",
            bbox_to_anchor=(bbox_to_anchor[0], bbox_to_anchor[1] - 0.1),
            ncol=5,
        )

    rs_factors_fs_entrainment(axs[0], ax_title_start_ix=0)
    rs_factors_rs_fraction(axs[1], ax_title_start_ix=3)
    rs_factors_rain_efficiency(axs[2], ax_title_start_ix=6)
    rs_factors_rs_entrainment_efficiency(axs[3], ax_title_start_ix=9)

    header_pad = 22
    headers.append(cm.plotting.add_row_header(
        axs[0][0],
        "FS tracer mixing\n" r"ratio $(\overline{q_{FS}})$ and" "\nupdraft mass " r" ($M$)",
        pad=header_pad, xycoords="subfigure fraction", xy=(0.09, 0.87), fontsize=22,
    ))
    headers.append(cm.plotting.add_row_header(
        axs[1][0], "RS tracer\n" r"fraction $(f_{RS})$",
        pad=header_pad, xycoords="subfigure fraction", xy=(0.09, 0.63), fontsize=22,
    ))
    headers.append(cm.plotting.add_row_header(
        axs[2][0], "Rain production\n" r"efficiency $(\eta_{RP})$",
        pad=header_pad, xycoords="subfigure fraction", xy=(0.09, 0.38), fontsize=22,
    ))
    headers.append(cm.plotting.add_row_header(
        axs[3][0], "RS tracer entrainment\n" r"efficiency $(\eta_{RSE}$)",
        pad=header_pad, xycoords="subfigure fraction", xy=(0.09, 0.14), fontsize=22,
    ))

    fig.suptitle(f"[{arm_label}]", fontsize=13, y=0.995)

    blt.save_figure_data(figure_data, figure_name)
    cm.plotting.fig_multisave(fig, figure_name, FIGURE_OUTPUT_DIR)
    return fig


for arm_label in RUN_DIRS_BY_ARM:
    with hu.blt_paths_for_arm(RUN_DIRS_BY_ARM[arm_label], figure_data_dir=FIGURE_DATA_OUTPUT_DIR):
        plot_figure_9_rs_entrainment_factors(arm_label, tracer_totals_dss_by_arm[arm_label])

## 2.3 Figure 10 — storm development statistics

Max updraft speed and updraft mass, and rainfall area by threshold. This is the figure most
directly sensitive to the initial-state changes, since CAPE feeds straight into updraft
strength — so it is the natural place to look first if a result moves.

In [ ]:
def plot_figure_10_storm_development(arm_label, development_dfs):
    """Render Figure 10 for one arm. Body carried over verbatim from the template."""
    figure_name = f"f10_storm_development_stats__{arm_label}"
    figure_data = {}

    fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(10, 8))
    fig.subplots_adjust(hspace=0.6, left=0.17, wspace=0.9, bottom=0.2)

    def ss_max_w(axs, ax_title_start_ix=0):
        updraft_axs = []
        for storm_ix, (storm, label) in enumerate(blt.STORM_LABELS.items()):
            ax = axs[storm_ix]
            development_df = development_dfs[storm]

            blt.plot_and_save(
                figure_data,
                f"storm-{storm}_max_updraft",
                ax.plot,
                development_df["t_minutes"],
                development_df["max_updraft"],
                label="Max updraft W",
                color=cm.plotting.get_nth_color(0),
            )
            updraft_twin_ax = ax.twinx()
            blt.plot_and_save(
                figure_data,
                f"storm-{storm}_updraft_mass",
                updraft_twin_ax.plot,
                development_df["t_minutes"],
                development_df["updraft_mass"],
                label="Updraft mass",
                color=cm.plotting.get_nth_color(1),
            )
            updraft_axs.append(updraft_twin_ax)
            # Empty line so the legend picks up the twin axis's series
            ax.plot([], [], color=cm.plotting.get_nth_color(1), label="Updraft mass")
            ax.set_xlabel("Simulation time (minutes)")
            ax.set_ylabel("Updraft speed (m/s)")
            ax.set_title(blt.axes_title(ax_title_start_ix + storm_ix, blt.STORM_LABELS[storm]))
            updraft_twin_ax.set_ylabel("Updraft mass (kg)")
        axs[1].legend(loc="center", bbox_to_anchor=(0.5, -0.35), ncol=2)

    def ss_rain(axs, ax_title_start_ix=0):
        for storm_ix, (storm, label) in enumerate(blt.STORM_LABELS.items()):
            ax = axs[storm_ix]
            development_df = development_dfs[storm]
            for threshold_ix in range(len(blt.RS_TRACER_THRESHOLD_RAIN_RATES)):
                blt.plot_and_save(
                    figure_data,
                    f"storm-{storm}_rain-{int(blt.RS_TRACER_THRESHOLDS_MM_HR[threshold_ix])}mmhr_area",
                    ax.plot,
                    development_df["t_minutes"],
                    # Gridpoint count -> square kilometres
                    development_df[f"rain_threshold{threshold_ix}"] * 500**2 / 1e6,
                    label=f"{blt.RS_TRACER_THRESHOLDS_MM_HR[threshold_ix]} mm/hr",
                    color=mpl.colormaps[blt.PROGRESSIVE_CMAP](threshold_ix / len(blt.RS_TRACERS)),
                )
            ax.set_xlabel("Simulation time (minutes)")
            ax.set_ylabel(r"Area ($km^{2}$)")
            ax.set_title(blt.axes_title(ax_title_start_ix + storm_ix, blt.STORM_LABELS[storm]))
        axs.flatten()[1].legend(loc="center", bbox_to_anchor=(0.5, -0.35), ncol=5)

    ss_max_w(axs[0])
    ss_rain(axs[1], ax_title_start_ix=3)

    cm.plotting.add_row_header(
        axs[0, 0], "Updraft max speed\n" r"and mass $(M)$",
        xycoords="figure fraction", xy=(0.14, 0.72), fontsize=22,
    )
    cm.plotting.add_row_header(
        axs[1, 0], "Rainfall area", xycoords="figure fraction", xy=(0.14, 0.25), fontsize=22
    )

    fig.suptitle(f"[{arm_label}]", fontsize=13, y=0.995)

    blt.save_figure_data(figure_data, figure_name)
    cm.plotting.fig_multisave(fig, figure_name, FIGURE_OUTPUT_DIR)
    return fig


for arm_label in RUN_DIRS_BY_ARM:
    with hu.blt_paths_for_arm(RUN_DIRS_BY_ARM[arm_label], figure_data_dir=FIGURE_DATA_OUTPUT_DIR):
        plot_figure_10_storm_development(arm_label, development_dfs_by_arm[arm_label])

## 2.4 Figures 11–12 — lofting to upper levels

### 2.4.1 Validate the coarse mass-flux estimator first

The manuscript's upward mass flux through 10 km comes from 2-second lite output. The
harmonized runs have none (`FRQLITE = 0`), so both arms must instead use a 5-minute
estimator. Before relying on it, measure how far it lands from the exact answer on the
**paper** runs, which do have lite output and an already-cached exact flux.

Five-minute sampling aliases the convective-scale vertical velocity field badly, so a large
discrepancy here would not be surprising. What matters is whether the discrepancy is
**similar across the three storms** — a common-mode bias leaves the cross-storm comparison
usable, a storm-dependent one does not.

In [ ]:
UPPER_LEVEL_Z_THRESHOLD_M = blt.UL_Z_THRESHOLD * 1000

# Coarse 5-minute estimator on the paper runs.
coarse_flux_dss_original = {
    storm_key: hu.calculate_cumulative_mass_flux_from_analysis_output(
        storm_ds,
        upper_level_z_threshold_m=UPPER_LEVEL_Z_THRESHOLD_M,
        output_interval_seconds=DELTA_T,
        horizontal_grid_spacing_m=DELTA_X,
    ).compute()
    for storm_key, storm_ds in storm_dss_by_arm[ORIGINAL_ARM_LABEL].items()
}

# The manuscript's exact 2-second flux, already cached from the paper analysis.
exact_flux_cache_path = blt.UL_FLUXES_DIR / f"ul_mass_fluxes_z-{blt.UL_Z_THRESHOLD}000.pkl"
print(f"Reading exact (2 s) fluxes from {exact_flux_cache_path}")
exact_flux_dss_original = pd.read_pickle(exact_flux_cache_path)

# Compare the two estimators at the end of the simulation, per storm.
flux_validation_rows = []
for storm_key, coarse_flux_ds in coarse_flux_dss_original.items():
    if storm_key not in exact_flux_dss_original:
        print(f"No exact flux cached for {storm_key}; skipping")
        continue
    exact_flux_ds = exact_flux_dss_original[storm_key]
    coarse_upward = float(coarse_flux_ds["cumu_upward_mass_flux"].isel(time=-1).values)
    exact_upward = float(exact_flux_ds["cumu_upward_mass_flux"].isel(time=-1).values)
    flux_validation_rows.append(
        {
            "storm": blt.STORM_LABELS[storm_key],
            "exact_2s_upward_kg": exact_upward,
            "coarse_5min_upward_kg": coarse_upward,
            "coarse_over_exact": coarse_upward / exact_upward,
        }
    )
flux_validation_table = pd.DataFrame(flux_validation_rows).set_index("storm")
display(HTML("<h4>Coarse (5 min) vs exact (2 s) upward mass flux, end of simulation</h4>"))
display(flux_validation_table)

# The decisive number: how much the coarse/exact ratio itself varies between storms.
# If this is small, the bias is common-mode and the cross-storm comparison survives.
_ratio_spread = flux_validation_table["coarse_over_exact"]
print(
    f"\ncoarse/exact ratio across storms: min={_ratio_spread.min():.4f}, "
    f"max={_ratio_spread.max():.4f}, "
    f"spread={100 * (_ratio_spread.max() / _ratio_spread.min() - 1):.2f}% "
    "(small => common-mode bias => Figures 11-12 remain comparable between arms)"
)

In [ ]:
# Coarse estimator for both arms, so the two are computed identically.
coarse_flux_dss_by_arm = {ORIGINAL_ARM_LABEL: coarse_flux_dss_original}
for arm_label in RUN_DIRS_BY_ARM:
    if arm_label in coarse_flux_dss_by_arm:
        continue
    coarse_flux_dss_by_arm[arm_label] = {
        storm_key: hu.calculate_cumulative_mass_flux_from_analysis_output(
            storm_ds,
            upper_level_z_threshold_m=UPPER_LEVEL_Z_THRESHOLD_M,
            output_interval_seconds=DELTA_T,
            horizontal_grid_spacing_m=DELTA_X,
        ).compute()
        for storm_key, storm_ds in tqdm(storm_dss_by_arm[arm_label].items())
    }

In [ ]:
# Upper-level tracer totals for both arms, cached per run so a rerun is cheap.
# Deliberately NOT restricted to the supercell's right mover, matching the manuscript.
upper_level_tracer_dfs_by_arm = {}

for arm_label, run_dirs in RUN_DIRS_BY_ARM.items():
    print(f"\n===== {arm_label} =====")
    arm_upper_level_dfs = {}
    for storm_key, storm_ds in tqdm(storm_dss_by_arm[arm_label].items()):
        cache_dir = hu.upper_level_derived_dir(run_dirs[storm_key], storm_key)
        cache_dir.mkdir(parents=True, exist_ok=True)
        cache_path = cache_dir / f"ul_tracer_totals_z-{UPPER_LEVEL_Z_THRESHOLD_M}.pkl"
        # hu.read_or_cache_dataframe rather than cm.utils.read_or_cache_to: the latter
        # cannot actually write .pkl files and silently no-ops the cache (see that
        # helper's docstring), so every rerun would recompute this from scratch.
        arm_upper_level_dfs[storm_key] = hu.read_or_cache_dataframe(
            cache_path,
            hu.calculate_upper_level_tracer_dataframe,
            storm_ds,
            UPPER_LEVEL_Z_THRESHOLD_M,
        )
    upper_level_tracer_dfs_by_arm[arm_label] = arm_upper_level_dfs

In [ ]:
# Precalculate the normalized quantities Figure 11 plots.
for arm_label, arm_upper_level_dfs in upper_level_tracer_dfs_by_arm.items():
    for storm_key, upper_level_df in arm_upper_level_dfs.items():
        cumulative_upward_flux = coarse_flux_dss_by_arm[arm_label][storm_key][
            "cumu_upward_mass_flux"
        ].values

        # Fixed-source tracer aloft, per unit of air lofted through the level
        upper_level_df["ul_fs_tracer_normalized-flux"] = (
            upper_level_df["total_fs_tracer_nc"].values / cumulative_upward_flux
        )

        for tracer in blt.RS_TRACERS:
            upper_level_df[f"ul_{tracer}_nc_normalized-flux"] = (
                upper_level_df[f"{tracer}_nc"].values / cumulative_upward_flux
            )
            # Normalize by the emitted amount from the full storm dataset rather than the
            # precomputed profiles, because the profiles cover only the supercell's right
            # mover while the amount emitted must be for the full domain.
            upper_level_df[f"ul_{tracer}_nc_normalized-acc"] = upper_level_df[
                f"{tracer}_nc"
            ].values / storm_dss_by_arm[arm_label][storm_key][f"ACC{tracer}_nc"].isel(
                {"z": 1}
            ).sum(["x", "y"])

In [ ]:
def plot_figure_11_upper_level_tracers(arm_label, upper_level_tracer_dfs):
    """Render Figure 11 for one arm.

    Rows 1 and 2 are normalized by the *coarse* upward mass flux rather than the
    manuscript's 2-second flux, for the reason given at the top of this section.
    Row 3 (normalized by the amount emitted) does not involve the flux at all and is
    therefore directly comparable to the manuscript.
    """
    figure_name = f"f11_tracer_entrainment_{blt.UL_Z_THRESHOLD}km__{arm_label}"
    cm.plotting.start_figure(figure_name, verbose=False)

    bbox_to_anchor = (0.5, -0.4)

    fig, axs = plt.subplots(ncols=3, nrows=3, figsize=(8, 12), sharex=True, sharey="row")
    fig.subplots_adjust(bottom=0.4, hspace=0.6)

    for storm_ix, (storm, this_ul_tracer_df) in enumerate(upper_level_tracer_dfs.items()):
        # Fixed-source
        axs[0, storm_ix].plot(
            this_ul_tracer_df["t_minutes"],
            this_ul_tracer_df["ul_fs_tracer_normalized-flux"].values,
            name=f"storm-{storm}_tracer-fstracernc_normalized-flux",
        )
        axs[0, storm_ix].set_title(blt.axes_title(storm_ix, blt.STORM_LABELS[storm]))

        for tracer_ix, tracer in enumerate(blt.RS_TRACERS):
            # Normalized by the air lofted through the level
            axs[1, storm_ix].plot(
                this_ul_tracer_df["t_minutes"],
                this_ul_tracer_df[f"ul_{tracer}_nc_normalized-flux"].values,
                color=mpl.colormaps[blt.PROGRESSIVE_CMAP](tracer_ix / len(blt.RS_TRACERS)),
                label=str(blt.RS_TRACER_THRESHOLDS_MM_HR[tracer_ix]) + " mm/hr",
                name=f"storm-{storm}_tracer-rstracer{tracer.lower()}_normalized-flux",
            )
            # Normalized by the amount emitted (flux-independent)
            axs[2, storm_ix].plot(
                this_ul_tracer_df["t_minutes"],
                this_ul_tracer_df[f"ul_{tracer}_nc_normalized-acc"].values,
                color=mpl.colormaps[blt.PROGRESSIVE_CMAP](tracer_ix / len(blt.RS_TRACERS)),
                label=str(blt.RS_TRACER_THRESHOLDS_MM_HR[tracer_ix]) + " mm/hr",
                name=f"storm-{storm}_tracer-rstracer{tracer.lower()}_normalized-acc",
            )

        axs[1, storm_ix].set_title(blt.axes_title(3 + storm_ix, blt.STORM_LABELS[storm]))
        axs[2, storm_ix].set_title(blt.axes_title(6 + storm_ix, blt.STORM_LABELS[storm]))

    cm.plotting.add_row_header(axs[0, 0], r"$N_{FS-UL}$/" + "\n" + r"$F_{upward}$", pad=80, fontsize=22)
    cm.plotting.add_row_header(axs[1, 0], r"$N_{RS-UL}$/" + "\n" + r"$F_{upward}$", pad=80, fontsize=22)
    cm.plotting.add_row_header(axs[2, 0], r"$N_{RS-UL}$/" + "\n" + r"$E_{RS}$", pad=80, fontsize=22)

    axs[0, 0].set_ylabel("#/kg")
    axs[0, 0].ticklabel_format(axis="y", style="sci", scilimits=(4, 4))
    axs[1, 0].set_ylabel("#/kg")
    axs[2, 0].set_ylabel("Fraction")

    for ax in axs.flatten():
        blt.label_x_axis(ax)
    for ax in axs[1:2].flatten():
        ax.set_yscale("log")
        ax.set_ylim([1e0, 1e5])

    axs[1, 1].legend(loc="center", bbox_to_anchor=(bbox_to_anchor[0], bbox_to_anchor[1] + 0.1), ncol=5)
    axs[2, 1].legend(loc="center", bbox_to_anchor=(bbox_to_anchor[0], bbox_to_anchor[1]), ncol=5)

    fig.suptitle(f"[{arm_label}] rows 1-2 use the coarse 5-min flux", fontsize=12, y=0.995)

    cm.plotting.save_figure_and_data(
        figure_output_dirs=[FIGURE_OUTPUT_DIR],
        data_output_dirs=[FIGURE_DATA_OUTPUT_DIR],
        extensions=[".pdf", ".png"],
    )
    return fig


for arm_label in RUN_DIRS_BY_ARM:
    with hu.blt_paths_for_arm(RUN_DIRS_BY_ARM[arm_label], figure_data_dir=FIGURE_DATA_OUTPUT_DIR):
        plot_figure_11_upper_level_tracers(arm_label, upper_level_tracer_dfs_by_arm[arm_label])

In [ ]:
def plot_figure_12_upper_level_fluxes(arm_label, coarse_flux_dss):
    """Render Figure 12 (cumulative up/down mass flux through 10 km) for one arm.

    Both curves use the coarse 5-minute estimator, so the absolute values will not match
    the manuscript's. Their *ratio* (below) is the quantity the manuscript actually quotes
    and is much less sensitive to the sampling interval.
    """
    fig, axs = plt.subplots(ncols=3, figsize=(9, 3.2), layout="constrained", sharex=True)
    for storm_ix, (storm, this_ul_flux_ds) in enumerate(coarse_flux_dss.items()):
        axs[storm_ix].plot(
            this_ul_flux_ds["t_minutes"], this_ul_flux_ds["cumu_upward_mass_flux"], label="Upward"
        )
        axs[storm_ix].plot(
            this_ul_flux_ds["t_minutes"], -this_ul_flux_ds["cumu_downward_mass_flux"], label="Downward"
        )
        axs[storm_ix].set_title(blt.axes_title(storm_ix, blt.STORM_LABELS[storm]))
        blt.label_x_axis(axs[storm_ix])
        axs[storm_ix].set_ylabel("kg")
    cm.plotting.clean_legend(axs[0], loc="upper left")
    fig.suptitle(f"[{arm_label}] cumulative mass flux through {blt.UL_Z_THRESHOLD} km (coarse 5-min)")
    cm.plotting.fig_multisave(fig, f"f12_ul_fluxes_{blt.UL_Z_THRESHOLD}km__{arm_label}", FIGURE_OUTPUT_DIR)
    return fig


for arm_label in RUN_DIRS_BY_ARM:
    with hu.blt_paths_for_arm(RUN_DIRS_BY_ARM[arm_label], figure_data_dir=FIGURE_DATA_OUTPUT_DIR):
        plot_figure_12_upper_level_fluxes(arm_label, coarse_flux_dss_by_arm[arm_label])

In [ ]:
# Net flux ratio at the end of the simulation: downward over upward. A ratio, so the
# coarse-sampling bias largely cancels.
net_flux_ratio_rows = []
for arm_label, coarse_flux_dss in coarse_flux_dss_by_arm.items():
    for storm_key, this_flux_ds in coarse_flux_dss.items():
        net_flux_ratio_rows.append(
            {
                "arm": arm_label,
                "storm": blt.STORM_LABELS[storm_key],
                "downward_over_upward": float(
                    (
                        this_flux_ds["cumu_downward_mass_flux"]
                        / -this_flux_ds["cumu_upward_mass_flux"]
                    )
                    .isel({"time": -1})
                    .values
                ),
            }
        )
display(HTML("<h4>Net flux ratio at end of simulation</h4>"))
display(pd.DataFrame(net_flux_ratio_rows).set_index(["arm", "storm"]).round(4))

## 2.5 Tracer entrainment statistics

The full statistics table from the manuscript notebook, per arm, followed by the
storm-to-storm ratio tables. The ratio tables are the ones to read: they are the numbers the
manuscript's comparative claims rest on.

In [ ]:
def calculate_tracer_statistics(tracer_totals_dss):
    """Compute the manuscript's full tracer entrainment statistics for one arm.

    Body carried over verbatim from `validation_comparison.ipynb`; only the data source
    is now an argument rather than a stashed global.
    """
    tracer_stats = {}
    tracer_stats_tracers = blt.RS_TRACERS + ["total_fs_tracer"]

    for storm_ix, (storm, this_tracer_ds) in enumerate(tracer_totals_dss.items()):
        # Limit to times when an updraft actually exists, or the per-kg means divide by zero
        this_tracer_ds = this_tracer_ds.where(this_tracer_ds["air_mass"] > 0)
        this_storm_stats = {}
        for ix, tracer in enumerate(tracer_stats_tracers):
            mean_mixing_ratio = this_tracer_ds[tracer + "_nc"] / this_tracer_ds["air_mass"]
            entrainment_efficiency = (
                this_tracer_ds[tracer + "_nc"] / this_tracer_ds["ACC" + tracer + "_nc"]
            )
            rain_production_efficiency = (
                this_tracer_ds["ACC" + tracer + "_nc"] / this_tracer_ds["air_mass"]
            )
            tracer_fraction = (
                this_tracer_ds[tracer + "_nc"] / this_tracer_ds["total_fs_tracer_nc"]
            )
            # Output is 5-minutely, so index 20 is t=100 min and index 36 is t=180 min
            this_post_100_ds = this_tracer_ds.isel({"time": slice(20, 37)})
            post_100_mixing_ratio = (
                this_post_100_ds[tracer + "_nc"] / this_post_100_ds["air_mass"]
            )

            this_storm_stats[tracer] = {
                "mean_mean": mean_mixing_ratio.mean(...).values,
                "max_mean": mean_mixing_ratio.max(...).values,
                "max_mean_time": this_tracer_ds["t_minutes"].values[
                    mean_mixing_ratio.argmax("time").values
                ],
                "eos_mean": mean_mixing_ratio.isel({"time": -1}).values,
                "max_mean_post_t100": post_100_mixing_ratio.max(...).values,
                "min_mean_post_t100": post_100_mixing_ratio.min(...).values,
                "mean_tracer_fraction": tracer_fraction.mean().values,
                "max_tracer_fraction": tracer_fraction.max().values,
                "max_tracer_fraction_time": this_tracer_ds["t_minutes"].values[
                    tracer_fraction.argmax("time").values
                ],
                "mean_rain_production_efficiency": rain_production_efficiency.mean().values,
                "max_rain_production_efficiency": rain_production_efficiency.max().values,
                "eos_rain_production_efficiency": rain_production_efficiency.isel(
                    {"time": -1}
                ).values,
                "integrated_rain_production_efficiency": rain_production_efficiency.sum().values,
                "t90_rain_production_efficiency": rain_production_efficiency.isel(
                    {"time": 90 // 5}
                ).values,
                "t100_rain_production_efficiency": rain_production_efficiency.isel(
                    {"time": 100 // 5}
                ).values,
                "t180_over_t75_rain_production_efficiency": (
                    rain_production_efficiency.isel({"time": 180 // 5})
                    / rain_production_efficiency.isel({"time": 75 // 5})
                ).values,
                "t180_over_t75_updraft_mass": (
                    this_tracer_ds["air_mass"].isel({"time": 180 // 5})
                    / this_tracer_ds["air_mass"].isel({"time": 75 // 5})
                ).values,
                "max_entrainment_efficiency": entrainment_efficiency.max().values,
                "max_entrainment_efficiency_time": this_tracer_ds["t_minutes"].values[
                    entrainment_efficiency.argmax("time").values
                ],
                "mean_entrainment_efficiency": entrainment_efficiency.mean().values,
                "integrated_entrainment_efficiency": entrainment_efficiency.sum().values,
                "eos_entrainment_efficiency": entrainment_efficiency.isel({"time": -1}).values,
                "t120_to_t180_mean_entrainment_efficiency": entrainment_efficiency.isel(
                    {"time": slice(24, 37)}
                ).mean().values,
                "t85_to_t180_mean_entrainment_efficiency": entrainment_efficiency.isel(
                    {"time": slice(17, 37)}
                ).mean().values,
            }
        tracer_stats[storm] = this_storm_stats
    return tracer_stats


tracer_stats_by_arm = {}
for arm_label in RUN_DIRS_BY_ARM:
    with hu.blt_paths_for_arm(RUN_DIRS_BY_ARM[arm_label], figure_data_dir=FIGURE_DATA_OUTPUT_DIR):
        tracer_stats_by_arm[arm_label] = calculate_tracer_statistics(
            tracer_totals_dss_by_arm[arm_label]
        )
        blt.save_figure_data(
            tracer_stats_by_arm[arm_label], f"entrainment_factor_stats__{arm_label}"
        )

    display(HTML(f"<h2>{arm_label}</h2>"))
    for storm_key, storm_stats in tracer_stats_by_arm[arm_label].items():
        display(HTML(f"<h3>{blt.STORM_LABELS[storm_key]}</h3>"))
        display(pd.DataFrame(storm_stats))

In [ ]:
# Storm-to-storm ratios of the full statistics table, side by side across arms.
# A ratio that is stable between arms means the comparative claim it supports is robust
# to the setup differences.
for comparison_pair in [
    ["sl_wk", "ic_wk"],
    ["sc_wk", "ic_wk"],
    ["sc_wk", "sl_wk"],
]:
    numerator_storm, denominator_storm = comparison_pair
    display(HTML(
        f"<h2>{blt.STORM_LABELS[numerator_storm]} / {blt.STORM_LABELS[denominator_storm]}</h2>"
    ))
    for arm_label, tracer_stats in tracer_stats_by_arm.items():
        display(HTML(f"<h4>{arm_label}</h4>"))
        display(
            pd.DataFrame(tracer_stats[numerator_storm])
            / pd.DataFrame(tracer_stats[denominator_storm])
        )

## 2.6 Cross-storm summary — the fairness test

This is what the notebook exists to produce.

The first table shows whether each storm's own numbers moved. That is interesting but not
decisive: a setup error that shifts all three storms by the same factor leaves every
comparative claim in the manuscript intact, because every one of them is a comparison.

The second table is the decisive one. It forms the ratio of each headline metric between
each pair of storms, in each arm, and reports how far that ratio moves. **A ratio that
survives harmonization means the cross-storm claim resting on it is robust to the setup
differences. A ratio that changes materially — or flips — means it is not.**

Rows whose change exceeds 10% are highlighted.

In [ ]:
# Reduce each arm to a small set of per-storm headline numbers.
headline_metrics_by_arm = {}
for arm_label in RUN_DIRS_BY_ARM:
    with hu.blt_paths_for_arm(RUN_DIRS_BY_ARM[arm_label], figure_data_dir=FIGURE_DATA_OUTPUT_DIR):
        headline_metrics_by_arm[arm_label] = hu.collect_headline_metrics(
            tracer_totals_dss_by_arm[arm_label],
            upper_level_tracer_dfs=upper_level_tracer_dfs_by_arm[arm_label],
            development_dfs=development_dfs_by_arm[arm_label],
        )
    display(HTML(f"<h4>Headline metrics — {arm_label}</h4>"))
    display(headline_metrics_by_arm[arm_label])

In [ ]:
# Did each storm's own numbers move?
display(HTML("<h3>Per storm: original vs harmonized</h3>"))
per_storm_comparison = hu.compare_arms_table(
    headline_metrics_by_arm, ORIGINAL_ARM_LABEL, HARMONIZED_ARM_LABEL
)
display(hu.style_percent_change(per_storm_comparison, "percent_change", threshold=10.0))

In [ ]:
# THE fairness test: did the ratios *between* storms move?
display(HTML("<h3>Cross-storm ratios: does the comparison between storms survive?</h3>"))
cross_storm_ratios = hu.cross_storm_ratio_table(headline_metrics_by_arm)
display(hu.style_percent_change(cross_storm_ratios, "ratio_percent_change", threshold=10.0))

# Compact read: the largest ratio movement per storm pair, which is the worst case for
# any comparative claim involving that pair.
display(HTML("<h4>Worst-case ratio movement per storm pair</h4>"))
display(
    cross_storm_ratios["ratio_percent_change"]
    .abs()
    .groupby("storm_pair")
    .agg(["max", "mean"])
    .round(2)
)

## 2.7 Sanity check: does the supercell right-mover mask still fit?

The supercell's analysis is restricted to its right mover by a **fixed geometric line**
(`blt.sc_wk_right_mover_boundary`, `y ≤ 78000 + x/5`), chosen by eye against the manuscript
run. A rerun supercell can split at a different time or place, in which case that fixed line
would clip the wrong part of the storm — and it would do so silently, because nothing in the
pipeline checks it.

This overlays the boundary on each arm's supercell condensate field at several times. The
right mover should sit below the line in both arms.

In [ ]:
# Overlay the right-mover boundary on the supercell's column-maximum condensate.
SNAPSHOT_TIME_INDICES = [12, 20, 28, 36]  # 5-minutely output, so t = 60, 100, 140, 180 min

fig, axs = plt.subplots(
    nrows=len(RUN_DIRS_BY_ARM),
    ncols=len(SNAPSHOT_TIME_INDICES),
    figsize=(4 * len(SNAPSHOT_TIME_INDICES), 3.6 * len(RUN_DIRS_BY_ARM)),
    layout="constrained",
    squeeze=False,
)

for arm_ix, arm_label in enumerate(RUN_DIRS_BY_ARM):
    supercell_ds = storm_dss_by_arm[arm_label]["sc_wk"]
    for time_ix, snapshot_time_index in enumerate(SNAPSHOT_TIME_INDICES):
        ax = axs[arm_ix, time_ix]
        # Column-maximum condensate makes the storm's footprint visible in one panel
        condensate_field = (
            supercell_ds["R_condensate"].isel({"time": snapshot_time_index}).max("z").compute()
        )
        ax.pcolormesh(
            supercell_ds["x"].values / 1000,
            supercell_ds["y"].values / 1000,
            condensate_field.transpose("y", "x").values,
            shading="auto",
        )
        # The fixed boundary the analysis mask uses
        boundary_x = supercell_ds["x"].values
        ax.plot(
            boundary_x / 1000,
            blt.sc_wk_right_mover_boundary(boundary_x) / 1000,
            color="red",
            linewidth=2,
            label="right-mover boundary",
        )
        ax.set_title(
            f"{arm_label}\nt = {int(supercell_ds['t_minutes'].isel(time=snapshot_time_index))} min",
            fontsize=11,
        )
        ax.set_xlabel("x (km)")
        if time_ix == 0:
            ax.set_ylabel("y (km)")

axs[0, 0].legend(loc="upper left", fontsize=9)
fig.suptitle("Supercell column-max condensate with the fixed right-mover boundary")
cm.plotting.fig_multisave(fig, "p2_supercell_right_mover_check", FIGURE_OUTPUT_DIR)

# Conclusions

**The supercell stops being a supercell on the squall line's profile.** This is a change in
storm mode, not merely in intensity.

### What changed

| metric | SC original | SC harmonized | change |
|---|---|---|---|
| peak updraft W (domain-wide, unmasked) | 41.3 m/s | 27.5 m/s | −33% |
| max updraft (4×4-min, time-smoothed) | 38.1 m/s | 19.9 m/s | −48% |
| mean RS mixing ratio | 14,367 | 1,175 | −92% |
| mean RS entrainment efficiency | 0.0431 | 0.0200 | −54% |
| UL RS per emitted | 0.0822 | 0.0040 | −95% |

The isolated convection and squall line are unchanged (not rerun), so all cross-storm ratio
movement here is the supercell moving: **65% mean, 99% max** for both SC/IC and SC/SL.

### The mechanism: the storm no longer splits

The fraction of 5 km updraft area falling inside the fixed right-mover mask, time-integrated:

| run | inside mask |
|---|---|
| SC original | **40.6%** |
| SC harmonized | **99.0%** |

The original splits into a right/left mover pair, so most of its updraft area lies outside the
right-mover mask by t=90 (60% → 34% by t=150). The harmonized run stays at 100% inside through
t=150 — it never produces a left mover. §2.7's figure shows this directly: two distinct cells
in the original by t=140, one weaker fragmented cluster in the harmonized run.

This also rules out the obvious artefact: the mask is **not** clipping the harmonized storm.
It captures essentially all of it, while the original's captures only the right mover by
design. If anything the harmonized numbers are the favourably biased ones, and they still show
84–99% less rain-sourced tracer entrainment.

### Why this is counterintuitive, and what it means

The harmonized supercell has *more* CAPE (2039 vs 1744 J/kg), similar CIN and LCL, and its own
unchanged wind profile — yet a third less updraft and no supercellular splitting. CAPE does not
explain it, so the profile *shape* is doing the work, most plausibly how the well-mixed boundary
layer was imposed (the IC/SL construction forces constant θ below 900 hPa; the supercell's was
built differently).

**What this does and does not establish.** This arm puts every storm on the *erroneous*
profile, so it shows the supercell is highly sensitive to which profile family it receives. It
does **not** show the manuscript's supercell result is wrong — the manuscript's supercell used
the correct profile. For what the manuscript should actually do, see the companion notebook
`harmonized_on_sc_profile.ipynb`, which corrects the two storms that were set up wrongly.

### The pressure anchoring is the culprit, not the moisture error

Decomposing the squall line's change using the `sl_wk/correct_wk_wellmixed` run (correct
profile, moisture bug **retained**), on mean rain-sourced entrainment efficiency:

| variant | value | vs paper |
|---|---|---|
| Paper (MSL anchoring + qv bug) | 0.0611 | — |
| Correct profile, qv bug kept | 0.00915 | **−85.0%** |
| Correct profile, qv fixed | 0.01002 | −83.6% |

The profile change accounts for essentially all of it; fixing the ~2% moisture error moves
the result back by only ~10% relative. The error that sounds like a technicality — anchoring
pressure at mean sea level rather than 1000 hPa, which shifts temperature ~1.1 K at fixed
height and arrives bundled with the unbalanced stratosphere — does the damage. The moisture
error, which sounds like it should matter most for a moisture-driven process, is second order.

### Caveats

- Ratios use the 10 mm/hr rain-sourced tracer as representative; §2.5's tables cover all five
  thresholds.
- Figures 11–12 use a 5-minute mass-flux estimator because the harmonized runs were written
  with `FRQLITE = 0`. §2.4.1 measures its bias against the exact 2-second flux on the paper
  runs: 1.9–4.7% high, spread across storms only 2.79%. That is small and near common-mode,
  so these comparisons hold.
- The harmonized isolated convection's updraft has fully dissipated by t=180, so its
  end-of-simulation in-updraft means are NaN. That is a property of the run, not a failure.